In [ ]:
import warnings; warnings.filterwarnings("ignore")
import torch, gc, os, json, re, time, random, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
import accelerate

random.seed(42); torch.manual_seed(42)
print(f"CUDA: {torch.version.cuda}  |  GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"
print("HF token set")

# GPU capability check (P100 sm_60 incompatible with PyTorch 2.13)
sm = torch.cuda.get_device_capability(0)
print(f"SM: {sm}")

USE_4BIT = False
BNB_CONFIG = None
if sm[0] >= 7:
    print("GPU compatible - attempting 4-bit...")
    try:
        os.system("pip install -q bitsandbytes --no-cache-dir 2>/dev/null")
        import bitsandbytes
        from transformers import BitsAndBytesConfig
        BNB_CONFIG = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        USE_4BIT = True
        print(f"bitsandbytes {bitsandbytes.__version__} - 4-bit ready")
    except Exception as e:
        print(f"4-bit unavailable: {e}")
else:
    print("GPU too old (sm<70). Skipping 4-bit. Using fp16 offload.")

print(f"PyTorch: {torch.__version__}  |  Accelerate: {accelerate.__version__}")
